# Configs For The Llama 3 Models

![chart](./showcase_images/model_sizes_chart.png)

- **Vocabulary Size:** The model's dictionary. It represents the words/tokens model can understand. It is static.
    - **Context Window length:** The model's short-term memory. Represents how much words/tokens the model can remember at one time. This changes over time, if you give the model content that is larger than its context window length, it forget the beginning of the content to make room for the end of the content. So, as you change with the model, it loses track of earlier context that no longer fits its context window.

In [20]:
# TODO add more bullet points above to describe the table.

In [ ]:
import os
import EasyJupyter
from pathlib import Path

class BaseConfig:
    """
    #TODO add docstring for what each attribute is for.

    The is the base config class that all models inherit from.
    """

    # TODO is max_seq_len needed?

    # All Llama models use the same below parameters, except for my scale down model.
    key_value_heads = 8
    activation_function = "SwiGLU"
    vocab_size = 128_000
    pos_embeddings = 500_000
    context_len = None #TODO

    # Force child classes to implement the below attributes
    _required_attributes: dict[str, type] = {
        "model_name": str,
        "num_layers": int,
        "d_model": int,
        "ffn_dim": int,
        "attn_heads": int,
        # "learning_rate" : float, #TODO add base learning rate
        "peak_lr": float,
        # "batch_size": int, #TODO add batch size
    }

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        for attr, expected_type in cls._required_attributes.items():
            value = getattr(cls, attr, None)
            if value is None:
                raise ValueError(f"Missing attribute {attr} in {cls.__name__}")
            if not isinstance(value, expected_type):
                raise TypeError(
                    f"Attribute {attr} in {cls.__name__} must be of type {expected_type}, but got {type(value)}"
                )
        pass

    # ================== Training ==================
    num_workers = 0 # 🚨 NOTE: For NVIDIA GPUs the Golden Rule is: num_worker = 4 * num_GPU | On Mac Silicone even though I have a 32 core GPU, it is still only one GPU, best to set num_workers = 0.
    num_epochs = 10
    continue_from_chpt:bool = False # Continue training the model from a check point
    # Checkpoint filename to load! 🚨 NOTE: The same config that was used to train the checkpoint must be used to load it!
    checkpoint_name:str = "" 

    # Overfit the the model on on example of the dataset, to see if model is working properly.
    train_overfitted_model:bool = False



    # ================== Dataset ==================
    # TODO remove any of the below commented out attributes if not needed.
    # pos_seq_len = 5000  # Positional Encoding sequence length
    # max_indiv_seq_len = 128 # Applies to individual sentences
    # max_batch_seq_tokens = 8_000 # Paper: 25_000. Applies sequence limit to an entire batch of sequences.
    # vocab_size = 37_000 # Limit to let tokenizer trainer know when to stop merging sub-words.
    # tokenizer = "BPE"
    # dataset_name = "WMT_2014_English_German"

    # # Percentage of database to download
    # perc_to_download: int = 1

    # total_sentence_pairs = None  # The total number of English-German sentence pairs, will be set later in code.
    special_tokens = { 
        "pad_token": "<|pad_id|>", 
        "doc_begin_token": "<|begin_of_doc|>", # The token for the beginning of the text.
        "doc_end_token": "<|end_of_doc|>", # The token for the end of document, it is injected between documents.
        "unk_token": "<|unk|>", # The unknown token .
    }



    def __init__(self):
        # ================== Folder Structure ==================
        self.PROJECT_ROOT = self._find_root()
        print("\nProject Root:", self.PROJECT_ROOT)
        self.DATA_DIR = self.PROJECT_ROOT / "data"
        self.MODEL_DIR = self.PROJECT_ROOT / "model"
        self.folders_to_make = [
            self.DATA_DIR,
            self.MODEL_DIR / "saved_models",
            self.MODEL_DIR / "saved_models" / "tokenizer",
            self.MODEL_DIR / "checkpoints",
        ]

    def _find_root(self) -> Path:
        """Find the root directory of the project, by climbing up the directory tree"""
        curr = Path.cwd().resolve()

        for parent in [curr] + list(curr.parents):
            # if (parent / ".gitignore").exists():
            #     return parent
            if parent.name == "How_to_build_an_LLM":
                return parent
            
        return curr

    def print_config(self):
        # TODO Make the print method in the parent and have the child call it to prints its own config.
        pass
    
    def _setup_folders(self):
        for folder in self.folders_to_make:
            folder.mkdir(parents=True, exist_ok=True)

In [16]:
class Llama3_8B(BaseConfig):
    """
    Llama 3.1 8B Configuration. This model is not multi-modal, it only takes in text!
    """

    model_name = "Llama 3.1 8B"
    num_layers = 32
    d_model = 4_096
    ffn_dim = 14_336
    attn_heads = 32
    peak_lr = 3e-4

    def __init__(self):
        super().__init__()

In [ ]:
class Llama3_scaled_down(BaseConfig):
    """
    Scaled down version of the Llama 3 architecture that is trainable on my Mac.
    """
    # TODO play around with the hyperparameters to see what my mac can handle.
    model_name = "Scaled down Llama 3"
    num_layers = 6
    d_model = 256
    ffn_dim = 1024
    attn_heads = 32
    peak_lr = 3e-4
    context_len = 256
    batch_size = 16

    # TODO for this model the basemodel configs need to be adjusted like vocab_size it to be somewhere between 16,000 to 32,000

    def __init__(self):
        super().__init__()
        print(os.getcwd())

In [18]:
# @i-c
l = Llama3_8B()
l._setup_folders()


Project root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM


In [19]:
# TODO add the config for the larger models